In [1]:
# %pip install -r requirements.txt

In [3]:
import os
import httpx
from dotenv import load_dotenv

# 1. Загружаем переменные из файла .env (ищет в текущей директории или выше)
load_dotenv()

# 2. Забираем настройки из системных переменных .env
API_KEY = os.getenv("LLM_PROVIDER_KEY")
BASE_URL = os.getenv("LLM_BASE_URL", "https://litellm.ai.nestle.ru/v1")
MODEL_NAME = os.getenv("LLM_MODEL", "gpt-4o-mini")


async def test_llm_connection():
    if not API_KEY:
        print("❌ Ошибка: LLM_PROVIDER_KEY не найден в переменных окружения (.env)!")
        return

    print(f"🔄 Проверяем подключение к LLM через {BASE_URL} (модель: {MODEL_NAME})...")

    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

    async with httpx.AsyncClient(timeout=30.0, verify=False) as client:
        try:
            response = await client.get(f"{BASE_URL.rstrip('/')}/models", headers=headers)
            print(f"Статус ответа: {response.status_code}")

            if response.status_code == 200:
                data = response.json()
                print("✅ Доступные модели:")
                # Обычно OpenAI-совместимые API возвращают модели в поле 'data'
                models = data.get("data", data)
                if isinstance(models, list):
                    for m in models:
                        model_id = m.get("id") if isinstance(m, dict) else m
                        print(f"  - {model_id}")
                else:
                    print(data)
            else:
                print(f"❌ Ошибка сервера: {response.text}")

        except Exception as e:
            print(f"❌ Ошибка при запросе: {e}")

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Ответь одним словом: Работает?"},
        ],
        "max_tokens": 100,
        "temperature": 0.0,
    }

    # Формируем корректный эндпоинт для Chat Completions
    endpoint = f"{BASE_URL.rstrip('/')}/chat/completions"

    try:
        # Используем httpx для асинхронного запроса без зависимости от openai SDK
        async with httpx.AsyncClient(timeout=15.0, verify=False) as client:
            response = await client.post(endpoint, json=payload, headers=headers)

            if response.status_code != 200:
                print(f"❌ Ошибка сервера [{response.status_code}]: {response.text}")
                return

            data = response.json()
            answer = data["choices"][0]["message"]["content"]
            usage = data.get("usage", {})
            print(data)

            print(f'✅ Успешно! Ответ модели: "{answer}"')
            if usage:
                print(f"📊 Использовано токенов: {usage.get('total_tokens')}")

    except Exception as e:
        print(f"❌ Ошибка при сетевом запросе:\n{type(e).__name__}: {e}")


# Запуск в ячейке Jupyter
await test_llm_connection()

🔄 Проверяем подключение к LLM через https://litellm.ai.nestle.ru/v1 (модель: NESTLE/Qwen3.8-27B-FP8)...
Статус ответа: 200
✅ Доступные модели:
  - public/DeepSeek-V4-Pro
  - openai/openai/whisper-large-v3
  - qwen3-coder-next
  - Nestle/whisper-turbo
  - Nestle/qwen-embed-06
  - NESTLE/Qwen3.8-27B-FP8
  - NESTLE/Qwen3.6-35b-a3B-fp8
  - deepseek-v4-pro
  - Nestle/qwen-rerank-06
  - PUBLIC/gemini-3.1-flash-lite
  - qwen3-embedding-0.6b
  - qwen3.5-397B-A17B
  - gpt-oss-120b
  - PUBLIC/claude-opus-4.8
{'id': 'chatcmpl-a8912dfc1bf76515', 'created': 1787252226, 'model': 'NESTLE/Qwen3.8-27B-FP8', 'object': 'chat.completion', 'system_fingerprint': 'vllm-0.25.1-596582ba', 'choices': [{'finish_reason': 'stop', 'index': 0, 'message': {'content': '\n\nДа', 'role': 'assistant', 'reasoning_content': 'We need answer user\'s request: "Ответь одним словом: Работает?" Russian: "Answer with one word: Works?" Need output one word. Likely "Да". Ensure final one word.\n', 'provider_specific_fields': {'reas